In [ ]:
import shutil
from pathlib import Path
from typing import cast

import lief
import pefile
import polars as pl

from addr_helpers import to_int_expr

# Paths

In [ ]:
old_iat_p = Path("iat.csv")

In [ ]:
base_to_exe = Path("../fake_neomon_host")
original_dump_path = base_to_exe / "NeoMon_dump.dll"
patched_path = base_to_exe / "NeoMon_patched_dump.dll"

# Patch IAT

## Load IAT

In [ ]:
iat2 = pl.read_csv(str(old_iat_p))
iat_seg = (
    iat2.sort("Calladdr")
    .fill_null("")
    .with_columns(
        (pl.col("Module") != pl.col("Module").shift(1)).cum_sum().alias("segment_id")
    )
    .fill_null(0)
    .filter(to_int_expr("Address") != 0)
    .filter(pl.col("Module") != "")
    .filter(~pl.col("Module").str.starts_with("section_"))
).drop("Address")
segments = [
    group.drop("segment_id")
    for _, group in iat_seg.group_by("segment_id", maintain_order=True)
]

lenseg = len(segments)
num_mods = iat_seg.unique("Module").unique().shape[0]
if lenseg == num_mods:
    print("Perfect: all obfuscated imports resolved")
elif lenseg < num_mods + 2:
    print("Good: most obfuscated imports resolved")
else:
    print(f"Number of segments: {lenseg}, number of unique modules: {num_mods}")

## Create IDT

In [ ]:
def create_32bit_ordinal_import(ordinal_number: int) -> lief.PE.ImportEntry:
    """
    Create a 32-bit import by ordinal

    Args:
        ordinal_number: The ordinal number (0-65535)
    """
    # Validate ordinal range
    if ordinal_number < 0 or ordinal_number > 0xFFFF:
        raise ValueError("Ordinal number must be between 0 and 65535")

    # For 32-bit PE:
    # - Set bit 31 to 1 (0x80000000)
    # - Bits 30-16 must be 0
    # - Bits 15-0 contain the ordinal
    ORDINAL_MASK_32 = 0x80000000
    data_value = ORDINAL_MASK_32 | ordinal_number

    # Create the import entry
    entry = lief.PE.ImportEntry(data_value, lief.PE.PE_TYPE.PE32)

    return entry

In [ ]:
shutil.copy(original_dump_path, patched_path)

pe_lief = cast(lief.PE.Binary, lief.PE.parse(patched_path))
print(hex(pe_lief.imagebase))
pe_lief.remove_all_imports()
pe_lief.optional_header.addressof_entrypoint = 0x13903

In [ ]:
for seg in segments:
    dll = seg["Module"][0]
    if dll is None or dll == "":
        continue

    mod = pe_lief.add_import(dll)
    for calladdr, ordinal, func, mname in seg.rows():
        if func.startswith("Ordinal#"):
            # ordinal = int(func.removeprefix("Ordinal#"))
            entry = create_32bit_ordinal_import(ordinal)
        else:
            entry = lief.PE.ImportEntry(func)
        mod.add_entry(entry)

In [ ]:
config = lief.PE.Builder.config_t()
config.imports = True

bb = lief.PE.Builder(pe_lief, config)
bb.build()
bb.write(str(patched_path))

## Reset IAT entries address to old IAT

In [ ]:
pe = pefile.PE(patched_path)
pe.full_load()

In [ ]:
assert len(pe.DIRECTORY_ENTRY_IMPORT) == lenseg, (  # type: ignore
    "Change the MAX_REPEATED_ADDRESSES to >20"
)

In [ ]:
for i, seg in enumerate(segments):
    first_thunk = int(seg["Calladdr"][0], 16)

    pe.DIRECTORY_ENTRY_IMPORT[i].struct.FirstThunk = (  # type: ignore
        first_thunk  # type: ignore
    )

In [ ]:
temp = "tmp"
pe.write(filename=temp)
pe.close()
shutil.move(temp, patched_path)